# 04 — Entraînement LoRA

Exécuter depuis la racine du dépôt. Les résultats ne sont valides que si les cellules sont réellement exécutées.

In [ ]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/'data').exists():
    ROOT=Path('/content/EduLab-AI-Version2')
print(ROOT)

In [ ]:
!pip -q install -r requirements-colab.txt

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'; OUT=ROOT/'models/edulab-teacher-qwen-0.5b-lora'
tok=AutoTokenizer.from_pretrained(MODEL_ID); tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
# Inspection réelle des modules
linear=sorted({n.split('.')[-1] for n,m in model.named_modules() if isinstance(m,torch.nn.Linear)})
print(linear)
target=[x for x in ['q_proj','k_proj','v_proj','o_proj'] if x in linear]; assert target
model=get_peft_model(model,LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,target_modules=target,task_type='CAUSAL_LM'))
def load(name):
 rows=[json.loads(x) for x in open(ROOT/f'data/processed/edulab_teacher_{name}.jsonl',encoding='utf-8')]
 texts=[tok.apply_chat_template([{'role':'user','content':r['instruction']+'\nContexte: '+r['context']},{'role':'assistant','content':r['response']}],tokenize=False) for r in rows]
 return Dataset.from_dict({'text':texts}).map(lambda b:tok(b['text'],truncation=True,max_length=384),batched=True,remove_columns=['text'])
train,val=load('train'),load('validation')
args=TrainingArguments(output_dir=str(OUT/'checkpoints'),num_train_epochs=2,learning_rate=2e-4,per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=8,fp16=torch.cuda.is_available(),eval_strategy='steps',save_strategy='steps',eval_steps=25,save_steps=25,load_best_model_at_end=True,metric_for_best_model='eval_loss',seed=20260723,report_to='none')
trainer=Trainer(model=model,args=args,train_dataset=train,eval_dataset=val,data_collator=DataCollatorForLanguageModeling(tok,mlm=False),callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
result=trainer.train(); metrics=trainer.evaluate(); model.save_pretrained(OUT,safe_serialization=True); tok.save_pretrained(OUT/'tokenizer')
(OUT/'metrics.json').write_text(json.dumps({**result.metrics,**metrics},indent=2)); (OUT/'training_args.json').write_text(json.dumps(args.to_dict(),indent=2,default=str))
print(metrics)